# Calculating network error with loss

The loss function is a function that is used to assess how wrong a model is. 


### Categorical cross-entropy
This is explicitly used to compare targets and some predicted distribution. This is mostly used with softmax activation function in output layer.

$L_i\ =\ -\ \sum_j\ y_{i,j}\ log(\hat{y}_{i,j})$

where i is the ith sample and j is the label/output index. Li is the sample loss value.

Cross entropy is a little different from logloss as logloss is applied to a binary classification problem.

Since for a sample i, only one of the j will be 1 (which is the correct label) and 0 for the rest, we can simply write this as

$L_i = -log(\hat{y}_{i,k})$

where k is an index of the target label.

In [ ]:
import numpy as np
from nnfs.datasets import spiral_data
import nnfs

In [ ]:

class Loss():
    def calculate(self, output, y):
        #calculate sample losses 
        sample_losses = self.forward(output, y)

        data_loss = np.mean(sample_losses)

        return data_loss

In [ ]:
class Loss_CategoricalCrossentropy(Loss):

    def forward(self, y_pred, y_true):

        samples = len(y_pred)

        #We are capping the y_pred for preventing log(0) or log(high value) both are not defined
        y_pred_clipped = np.clip(y_pred, 1e-7, 1- 1e-7)

        if len(y_true.shape) == 1:
            correct_confidences = y_pred_clipped[range(samples), y_true]

        if len(y_true.shape) == 2:
            correct_confidences = np.sum(y_pred_clipped * y_true, axis = 1)

        negative_log_likelihood = -np.log(correct_confidences)
        return negative_log_likelihood


In [ ]:
#Example

softmax_output = np.array([[0.7, 0.1, 0.2],
                          [0.1, 0.5, 0.4],
                          [0.02, 0.9, 0.08]])

class_targets = np.array([[1, 0, 0],
                          [0, 1, 0],
                          [0, 1, 0]])

loss_function = Loss_CategoricalCrossentropy()

loss = loss_function.calculate(softmax_output, class_targets)

In [ ]:
x = [-np.log(0.7) , -np.log(0.5) , -np.log(0.9)]

In [ ]:
x

In [ ]:
np.mean(x)

Now we can use it on the spiral data

In [ ]:
X, y = spiral_data(samples = 100, classes = 3)

class Dense_layer:

    def __init__(self, n_inputs, n_neurons):
        self.weights = 0.01 * np.random.randn(n_inputs, n_neurons)
        self.biases = np.zeros((1, n_neurons))
    
    def forward(self, inputs):
        self.output = np.dot(inputs, self.weights) + self.biases


class Activation_Relu:

    def forward(self, inputs):
        self.output = np.maximum(0, inputs)


class Activation_softmax:
    
    def forward(self, inputs):

        exp_values = np.exp(inputs - np.max(inputs, axis = 1, keepdims = True))
        probabilities = exp_values/np.sum(exp_values, axis = 1, keepdims = True)
        self.output = probabilities

import numpy as np
class Loss():
    def calculate(self, output, y):
        #calculate sample losses 
        sample_losses = self.forward(output, y)

        data_loss = np.mean(sample_losses)

        return data_loss
    
class Loss_CategoricalCrossentropy(Loss):

    def forward(self, y_pred, y_true):

        samples = len(y_pred)

        #We are capping the y_pred for preventing log(0) or log(high value) both are not defined
        y_pred_clipped = np.clip(y_pred, 1e-7, 1- 1e-7)

        if len(y_true.shape) == 1:
            correct_confidences = y_pred_clipped[range(samples), y_true]

        if len(y_true.shape) == 2:
            correct_confidences = np.sum(y_pred_clipped * y_true, axis = 1)

        negative_log_likelihood = -np.log(correct_confidences)
        return negative_log_likelihood


In [ ]:
#Creating a dense layer with 3 neuron and 2 input features means 2 weights in each neuron
dense1 = Dense_layer(2,3)

# Making a forward pass in this layer
dense1.forward(X)

# applying a ReLU activation function to the output of the first layer
activation1 = Activation_Relu()

activation1.forward(dense1.output)

# Creating another layer with 3 neuron and 3 input layers
dense2 = Dense_layer(3,3)

#Making a forward pass in 2nd layer with output from 1st layer
dense2.forward(activation1.output)

#Applying softmax activation to the output
activation2 = Activation_softmax()

activation2.forward(dense2.output)

#Loss function
loss_function = Loss_CategoricalCrossentropy()

loss = loss_function.calculate(activation2.output, y)
print(activation2.output[:5])
print(loss)

# Acuuracy

Accuracy is another metric commenly used. Accuracy is simply comparing target with estimated prediction (y_true == y_pred). 2 models with same accuracy can have different loss.


In [ ]:
predictions = np.argmax(activation2.output, axis = 1)

if len(y.shape) == 2:
    y = np.argmax(y, axis = 1)

accuracy = np.mean(predictions == y)

print(accuracy)